In [6]:
from evalplus.data import get_human_eval_plus, get_mbpp_plus
import requests, json, os
import pynvml, time, threading

path = "../data/processed/pass@1/qwen_3b"
humaneval_filename = "humaneval_solutions.jsonl"
mbpp_filename = "mbpp_solutions.jsonl"
ollama_url = "http://host.docker.internal:11434/api/generate"

In [7]:
monitoring = False
energy = 0.0
peak_vram = 0.0

In [8]:
def monitor_GPU(handle):
	global energy, peak_vram, monitoring
	power_supported = True
	last_time = time.time()
	while monitoring:
		now = time.time()
		time_diff = now-last_time
		last_time = now

		if power_supported:
			try:
				watt = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0 # function returns mW
				energy += watt * time_diff
			except pynvml.NVMLError_NotSupported:
				power_supported = False

		memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
		vram = memory.used / (1024 ** 2)
		peak_vram = vram if vram > peak_vram else peak_vram

		time.sleep(0.1)

In [9]:
def generate_solutions(problems, output_file):
    global energy, peak_vram, monitoring


    if not os.path.exists(path):
        os.makedirs(path)

    completed_tasks = set()
    if os.path.exists(output_file):
        with open(output_file, "r") as f:
            for line in f:
                data = json.loads(line.strip())
                completed_tasks.add(data["task_id"])

    if(len(problems) - len(completed_tasks)) == 0:
        print("problems solutions already generated.\n")
        return

    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0) # takes the first GPU
    energy = 0.0
    peak_vram = 0.0
    monitoring = True
    
    with open(output_file, "a") as f:
        thread = threading.Thread(target=monitor_GPU, args=(handle,))
        thread.start()
        start_time = time.time()

        for task_id in problems.keys():
            if(task_id in completed_tasks): continue
            problem = problems[task_id]
            prompt = problem["prompt"]

            payload = {
                "model": "qwen2.5-coder:3b",
                "prompt": prompt,
                "stream": False
            }
            response = requests.post(ollama_url, json=payload).json()
            generated_code = response.get("response", "")

            
            f.write(json.dumps({
                "task_id": task_id,
                "solution": generated_code
            }) + "\n")

        monitoring = False
        thread.join()
        total_time = time.time()-start_time
        print(f"time: {total_time}\nenergy: {energy}\npeak_vram: {peak_vram}\n")
        pynvml.nvmlShutdown()

In [10]:

humaneval_problems = get_human_eval_plus()
print(f"Number of problems: {len(humaneval_problems)}")
generate_solutions(humaneval_problems, f"{path}/{humaneval_filename}")


Number of problems: 164
time: 860.4229180812836
energy: 0.0
peak_vram: 3171.98828125



In [11]:
mbpp_problems = get_mbpp_plus()
print(f"Number of problems: {len(mbpp_problems)}")
generate_solutions(mbpp_problems, f"{path}/{mbpp_filename}")

Number of problems: 378
time: 1651.9043629169464
energy: 0.0
peak_vram: 3153.14453125



The results contained natural language explantations which made the tests fail. So we cleaned it with the following code:

In [12]:
def clean_file(file):
    import re

    regex = r"```python\n(.*?)\n```"

    outputsamples = []
    with open(file, "r") as f:
        for line in f:
            sample = json.loads(line)
            solution = sample['solution']
            match = re.search(regex, solution, re.DOTALL)
            solution = match.group(1) if match else solution
            sample['solution'] = solution
            outputsamples.append(sample)


    with open(file, "w") as f:
        for sample in outputsamples:
            f.write(json.dumps(sample)+"\n")

In [13]:
clean_file(f"{path}/{humaneval_filename}")
clean_file(f"{path}/{mbpp_filename}")

In [17]:
!evalplus.evaluate --dataset humaneval --samples ../data/processed/pass@1/qwen_3b/humaneval_solutions.jsonl
!evalplus.evaluate --dataset mbpp --samples ../data/processed/pass@1/qwen_3b/mbpp_solutions.jsonl

Load from previous results from ../data/processed/pass@1/qwen_3b/humaneval_solutions_eval_results.json
humaneval (base tests)
pass@1:	0.787
humaneval+ (base + extra tests)
pass@1:	0.744
Computing expected output...
Expected outputs computed in 49.26s
Reading samples...
378it [00:00, 2338.70it/s]
 99%|███████████████████████████████████████▍| 373/378 [00:39<00:00, 21.62it/s]/home/vscode/.local/lib/python3.11/site-packages/evalplus/evaluate.py:239: UserWarning: No samples had finished testing in the last 20s
  warn("No samples had finished testing in the last 20s")
/home/vscode/.local/lib/python3.11/site-packages/evalplus/evaluate.py:240: UserWarning: 1 samples to be tested: {'Mbpp/765 (line 342 in ../data/processed/pass@1/qwen_3b/mbpp_solutions.jsonl)'}
  warn(f"{len(remainings)} samples to be tested: {remainings}")
100%|████████████████████████████████████████| 378/378 [01:27<00:00,  4.32it/s]
mbpp (base tests)
pass@1:	0.672
mbpp+ (base + extra tests)
pass@1:	0.582
